In [ ]:
import os
import shutil
import zipfile
from pathlib import Path
import pandas as pd

In [ ]:
EXCEL_PATH = "/content/sample_data/Exelfile.xlsx"
ZIP_PATH   = "/content/sample_data/Zipfile.zip"
OUTPUT_DIR = "/content/sample_data/renamed_output"


In [ ]:
df = pd.read_excel(EXCEL_PATH)


In [ ]:
# Clean column names (strip spaces and stray * characters)
df.columns = [c.strip().rstrip("*").strip() for c in df.columns]
print("Excel loaded")
print(f"   Columns  : {list(df.columns)}")
print(f"   Questions: {len(df)}\n")

Excel loaded
   Columns  : ['Display Order', 'QBG Subject Id', 'QBG Chapter Id', 'QBG Topic Id', 'QBG SubTopic Id', 'Question Text', 'Question Image', 'Option Text', 'Option Image', 'Answer', 'Sol Text', 'Sol Image', 'Sol Video', 'Question Type', 'Positive Marks', 'Negative Marks', 'Partial Marks', 'Difficulty level', 'Language', 'Comp Id', 'Comp Image', 'Section Image', 'QBG Question id', 'QR Link', 'PYQ Years']
   Questions: 45



In [ ]:
#Build the mapping dictionary
mapping = {}

for _, row in df.iterrows():

    # Get the question number for this row
    n = int(row["Display Order"])

    # Get the random question image filename
    q_img = str(row["Question Image"]).strip()

    if q_img and q_img != "nan":

        # Map  QUES_ENG_<random>.png  →  Q<n>.png
        ext = Path(q_img).suffix or ".png"
        mapping[q_img] = f"Q{n}{ext}"

        # Derive the solution filename by swapping the prefix
        # Map  SOLU_ENG_<random>.png  →  S<n>.png
        sol_img = q_img.replace("QUES_ENG_", "SOLU_ENG_")
        mapping[sol_img] = f"S{n}{ext}"

    # Also handle Sol Image column if it is filled (future-proofing)
    if "Sol Image" in df.columns and pd.notna(row.get("Sol Image")):
        s = str(row["Sol Image"]).strip()
        if s and s != "nan":
            ext = Path(s).suffix or ".png"
            mapping[s] = f"S{n}{ext}"   # overrides derived name if different

print(f" Mapping built: {len(mapping)} entries")
print("   Preview (first 6 entries):")
for orig, new in list(mapping.items())[:6]:
    print(f"      {orig}  →  {new}")
print()

 Mapping built: 90 entries
   Preview (first 6 entries):
      QUES_ENG_fggct35vp5l1m8gyjj540mdzs.png  →  Q1.png
      SOLU_ENG_fggct35vp5l1m8gyjj540mdzs.png  →  S1.png
      QUES_ENG_yh0nkd2ywd9bfp9utd4og9xz2.png  →  Q2.png
      SOLU_ENG_yh0nkd2ywd9bfp9utd4og9xz2.png  →  S2.png
      QUES_ENG_ms799v5gsxaej8rbep9rool8y.png  →  Q3.png
      SOLU_ENG_ms799v5gsxaej8rbep9rool8y.png  →  S3.png



In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

renamed = 0   # count of successfully renamed files
kept    = 0   # count of files with no mapping (kept original name)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:

    all_entries = zf.namelist()
    print(f"✅ ZIP opened: {len(all_entries)} files found inside\n")

    for entry in all_entries:

        # Get just the filename (ignore any subfolder paths inside ZIP)
        basename = Path(entry).name
        if not basename:
            continue   # skip folder entries

        # Look up the new name from our mapping
        new_name = mapping.get(basename)

        # Write the file to the output folder with its new name
        dest = Path(OUTPUT_DIR) / (new_name if new_name else basename)

        with zf.open(entry) as src, open(dest, "wb") as dst:
            shutil.copyfileobj(src, dst)

        # Log what happened
        if new_name:
            print(f"  ✓  {basename}  →  {new_name}")
            renamed += 1
        else:
            print(f"  ?  {basename}  →  kept as-is (no mapping found)")
            kept += 1


✅ ZIP opened: 90 files found inside

  ✓  QUES_ENG_1d89mje8d66oslloxpqrp89zx.png  →  Q8.png
  ✓  QUES_ENG_1mplbs0a1ef3tk9ajt7dekcrz.png  →  Q5.png
  ✓  QUES_ENG_21mse4bc9esrt1ttx27hmb6eh.png  →  Q16.png
  ✓  QUES_ENG_3n7uedijrhvmheudh8tcknnkf.png  →  Q27.png
  ✓  QUES_ENG_3v8hlekq3nfk2jr19gh1vs6b5.png  →  Q40.png
  ✓  QUES_ENG_4eql6j6i62f839yuizs4zde6m.png  →  Q45.png
  ✓  QUES_ENG_5ma4owhym52k8b1bi4ff1zu5y.png  →  Q7.png
  ✓  QUES_ENG_6ubdbco1q6lab5vvc69u8zxsa.png  →  Q44.png
  ✓  QUES_ENG_7go86b7t0dc7tp5fi5sy971qq.png  →  Q26.png
  ✓  QUES_ENG_7q1vpg8vrvr546hqgj15et3wo.png  →  Q29.png
  ✓  QUES_ENG_9n8q8klvjm0m7kam0eh8el1on.png  →  Q14.png
  ✓  QUES_ENG_axsmxp57v00jnlju36fhcfw2y.png  →  Q20.png
  ✓  QUES_ENG_b6crb4k23pcrshcrt32625crg.png  →  Q12.png
  ✓  QUES_ENG_cr7zp2d949rdr0kzc4f031u2s.png  →  Q33.png
  ✓  QUES_ENG_edh8q2p8cu6tvoigmmoipgn7y.png  →  Q28.png
  ✓  QUES_ENG_enilhf6ttgffqv8hyo7lyo95n.png  →  Q31.png
  ✓  QUES_ENG_fggct35vp5l1m8gyjj540mdzs.png  →  Q1.png
  ✓  QUES_ENG_h

In [ ]:

# STEP 3 — Summary and verification


print(f"\n{'='*60}")
print(f"✅ Done!")
print(f"   {renamed} files renamed  |  {kept} kept as-is")
print(f"   Output folder : {OUTPUT_DIR}")
print(f"{'='*60}\n")

# List what's in the output folder, split by Q and S files
all_output = sorted(os.listdir(OUTPUT_DIR))
q_files    = sorted([f for f in all_output if f.startswith("Q")],
                     key=lambda x: int(x[1:].split(".")[0]))
s_files    = sorted([f for f in all_output if f.startswith("S")],
                     key=lambda x: int(x[1:].split(".")[0]))

print(f"Question images ({len(q_files)}):")
print("  " + "  ".join(q_files))
print()
print(f"Solution images ({len(s_files)}):")
print("  " + "  ".join(s_files))


✅ Done!
   90 files renamed  |  0 kept as-is
   Output folder : /content/sample_data/renamed_output

Question images (45):
  Q1.png  Q2.png  Q3.png  Q4.png  Q5.png  Q6.png  Q7.png  Q8.png  Q9.png  Q10.png  Q11.png  Q12.png  Q13.png  Q14.png  Q15.png  Q16.png  Q17.png  Q18.png  Q19.png  Q20.png  Q21.png  Q22.png  Q23.png  Q24.png  Q25.png  Q26.png  Q27.png  Q28.png  Q29.png  Q30.png  Q31.png  Q32.png  Q33.png  Q34.png  Q35.png  Q36.png  Q37.png  Q38.png  Q39.png  Q40.png  Q41.png  Q42.png  Q43.png  Q44.png  Q45.png

Solution images (45):
  S1.png  S2.png  S3.png  S4.png  S5.png  S6.png  S7.png  S8.png  S9.png  S10.png  S11.png  S12.png  S13.png  S14.png  S15.png  S16.png  S17.png  S18.png  S19.png  S20.png  S21.png  S22.png  S23.png  S24.png  S25.png  S26.png  S27.png  S28.png  S29.png  S30.png  S31.png  S32.png  S33.png  S34.png  S35.png  S36.png  S37.png  S38.png  S39.png  S40.png  S41.png  S42.png  S43.png  S44.png  S45.png


In [ ]:

# STEP 4 — (Optional) Zip the output for easy download


output_zip = "/content/sample_data/renamed_output.zip"

with zipfile.ZipFile(output_zip, "w") as zout:
    for f in all_output:
        zout.write(os.path.join(OUTPUT_DIR, f), arcname=f)

print(f"\n✅ Output zipped → {output_zip}")
print("   Download it from the Colab file browser (left panel → Files)")


✅ Output zipped → /content/sample_data/renamed_output.zip
   Download it from the Colab file browser (left panel → Files)


end